#### Integration of Apache Kafka and Spark Streaming

Let's give some (admittedly odd) context to this notebook. An alien entity has entered Portugal and suddenly starts turning citizens into scotsmen! Fear grips the country. We need to find out what is happening and how it is affecting the population.

[To understand what is going on](https://www.youtube.com/watch?v=qxDJMn-534Y)
(This is a Monty **Python** sketch)

We have been charged by the Portuguese government to analyze how the population has already changed. We will use **Spark Streaming** to stream data from our Kafka cluster and analyze live-streamed epidemiological data.

We will work with streams, stream-stream joins and stream-static joins.

Stream-stream joins involve combining two continuous flows of data, which is more complex because events from both sources may arrive at different times or out of order. To make this work, Spark must temporarily store the data from both streams to check for matches, within, for exampke, a 5-minute window. To prevent the system from running out of memory, you must define "watermarks" and time constraints, which tell the engine how long it needs to wait for late data before it can safely discard old records.

Stream-static joins are the simpler of the two join types, used primarily to enrich real-time data with fixed reference information. This allows you to add context to your streaming data without needing to manage complex state for the static side of the join.

<img src="img/scottish_portugal.png" height="500" width="700"/>

Structured Streaming treats a ``stream`` of data as a table that is updated in real time. An underlying process then regularly checks for updates and updates the table, if necessary. The API around Structured Streaming is designed in such a way that what works on your DataFrame, should also work on your streamed DataFrame! 

``Spark Streaming`` is a subset of Spark's functionalities that allows us to work with event-based data, as with our Kafka cluster. We set some global variables and import Schema Types to **structure our data**.

In [1]:
import pyspark.sql.functions as F

from pyspark.sql.types import StructType, StringType, DoubleType, StructField, IntegerType, TimestampType
from pyspark.sql import SparkSession

KAFKA_BOOTSTRAP_SERVERS = "localhost:8098"
KAFKA_TOPIC = "scotsmen"

We initialize a Spark Session. We import the ``Spark SQL Kafka Connector`` as a dependency. 

In [2]:
# Initialize local spark session
spark = SparkSession \
    .builder \
    .appName("kafka_streaming") \
    .config("spark.streaming.stopGracefullyOnShutdown", True) \
    .config('spark.jars.packages', 'org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0') \
    .config("spark.sql.shuffle.partitions", 4) \
    .master("local[*]") \
    .getOrCreate()

:: loading settings :: url = jar:file:/system/conda/miniconda3/envs/cloudspace/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/zeus/.ivy2/cache
The jars for the packages stored in: /home/zeus/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-fff8c5a7-48cf-4434-a980-69b95b2ce69d;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.0 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.3 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report :: resolve 576ms :: artifacts dl 19ms
	:: mod

Using the ``subscribe-publish`` paradigm, we subscribe to the Kafka topic ``scotsmen``.

In [3]:
# Read from ``KAFKA_TOPIC``
streaming_df = spark.readStream.format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS) \
    .option("subscribe", KAFKA_TOPIC) \
    .option("startingOffsets", "earliest") \
    .load()    

In [4]:
# Instantiate the schema of the messages received
scotsmen_schema = StructType([
  StructField("district", StringType()),
  StructField("new_scotsmen", IntegerType()),
  StructField("timestamp", TimestampType())
])

In [5]:
# We select the 'value' column and cast it as a String
json_df = streaming_df.select(
    F.from_json(F.col("value").cast("string"), scotsmen_schema).alias("value"), 
    "timestamp"
    )

In [6]:
# We instantiate an SQL view to inspect our data
json_df.select("value.*").createOrReplaceTempView("scotsmen")

In [9]:
# # Sample query from ``scotsmen`` table
# scotsmen_query = spark.sql("SELECT * FROM scotsmen")

# query = scotsmen_query.writeStream.toTable("my_table")

Note that, as with the non-streaming API, there are ``transformations`` and ``actions``. Execution of a query operation on Spark Streaming is lazy.

### Input Sources & Sinks

Spark Structured Streaming supports different input sources and sinks. Supported sinks are:
1. Kafka Streams
2. Files on a distributed file system (HDFS, S3). Spark will read files from a directory
3. A Socket Source

While input sources specify the origin of the data, sinks specify where the data will be written. Those sinks can be:
1. Kafka sink: Pushes data to Kafka
2. Files sink: Writes the output to a file (JSON, parquet, CSV etc.)
3. ForEach sink: Can be used to for each row of a DataFrame for custom storage logic
4. Console sink: Used for testing
5. Memory: Used for debugging

``Memory`` and ``Console``sinks are very similar. ``Memory`` mode makes the data available in an in-memory table for interactive inspection.

In [10]:
# This will print the results to the console
query = scotsmen_query.writeStream.outputMode("append").format("console").start()

25/11/28 16:24:04 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-1e591211-dd18-4cf4-a1db-0c089a6ce9d2. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
25/11/28 16:24:04 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


25/11/28 16:24:04 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


-------------------------------------------
Batch: 0
-------------------------------------------
+--------+------------+---------+
|district|new_scotsmen|timestamp|
+--------+------------+---------+
+--------+------------+---------+



In [17]:
# Stopping the query
query.stop()

There are three different output modes available. Here, we used ``append``, which only adds new records to the sink. The other two are ``update`` and ``complete``. ``update`` mode updates the data in the sink, while ``complete`` mode replaces the data in the sink.

In [11]:
# Write this query to a memory table
memory_query = scotsmen_query \
    .writeStream \
    .outputMode("append") \
    .queryName("scotsmen_table") \
    .format("memory") \
    .start()

25/11/28 16:24:25 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-cb247430-30a5-41e9-911f-38703cd26942. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
25/11/28 16:24:25 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


25/11/28 16:24:25 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


-------------------------------------------
Batch: 1
-------------------------------------------


+--------+------------+-------------------+
|district|new_scotsmen|          timestamp|
+--------+------------+-------------------+
|Braganca|        1444|2025-11-28 16:26:19|
+--------+------------+-------------------+



-------------------------------------------
Batch: 2
-------------------------------------------
+--------+------------+-------------------+
|district|new_scotsmen|          timestamp|
+--------+------------+-------------------+
|  Leiria|         941|2025-11-28 16:26:23|
+--------+------------+-------------------+



-------------------------------------------
Batch: 3
-------------------------------------------
+--------------+------------+-------------------+
|      district|new_scotsmen|          timestamp|
+--------------+------------+-------------------+
|Castelo Branco|         428|2025-11-28 16:26:26|
+--------------+------------+-------------------+

-------------------------------------------
Batch: 4
-------------------------------------------
+----------------+------------+-------------------+
|        district|new_scotsmen|          timestamp|
+----------------+------------+-------------------+
|Viana do Castelo|        1450|2025-11-28 16:26:31|
+----------------+------------+-------------------+



In [12]:
spark.sql("SELECT district, sum(new_scotsmen) AS total_new_scotsmen FROM scotsmen_table GROUP BY district").show()

+----------------+------------------+
|        district|total_new_scotsmen|
+----------------+------------------+
|        Braganca|              1444|
|          Leiria|               941|
|  Castelo Branco|               428|
|Viana do Castelo|              1450|
+----------------+------------------+



#### Window Functions

In [7]:
# Convert JSON to DataFrame
new_scotsmen_df = json_df.select(
    F.col("value.district").alias("conversion_district"),
    F.col("value.timestamp").alias("conversion_timestamp"),
    F.col("value.new_scotsmen").alias("new_scotsmen")
)

In [8]:
# Watermarking ensures ensures that late events 
# (up to 30 seconds after their event timestamp) 
# are considered in the aggregation, but any event arriving after that will be ignored.
windowed_df = new_scotsmen_df.withWatermark("conversion_timestamp", "30 seconds") \
    .groupBy(
        F.window(new_scotsmen_df["conversion_timestamp"], "3 minute"),  # 3-minute window
        new_scotsmen_df["conversion_district"]  # Group by district
    ) \
    .agg(
        F.count("*").alias("event_count"),   # Count events in each window for each district
        F.sum("new_scotsmen").alias("total_new_scotsmen"),  # Sum of values for each district in each window
        F.avg("new_scotsmen").alias("average_new_scotsmen") # Compute the average value for each district in each window
    )

In [16]:
window_query = windowed_df \
    .writeStream \
    .outputMode("complete") \
    .queryName("new_scotsmen_aggregated") \
    .format("memory") \
    .start()

25/11/28 16:26:53 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-c7551a0a-8fcb-4921-878b-d564290ae651. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
25/11/28 16:26:53 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


25/11/28 16:26:53 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


-------------------------------------------
Batch: 8
-------------------------------------------
+--------+------------+-------------------+
|district|new_scotsmen|          timestamp|
+--------+------------+-------------------+
|  Guarda|         448|2025-11-28 16:26:57|
+--------+------------+-------------------+

-------------------------------------------
Batch: 9
-------------------------------------------
+--------+------------+-------------------+
|district|new_scotsmen|          timestamp|
+--------+------------+-------------------+
| Setubal|        1091|2025-11-28 16:27:01|
+--------+------------+-------------------+



In [18]:
spark.sql("SELECT * FROM new_scotsmen_aggregated").show()

+--------------------+--------------------+-----------+------------------+--------------------+
|              window| conversion_district|event_count|total_new_scotsmen|average_new_scotsmen|
+--------------------+--------------------+-----------+------------------+--------------------+
|{2025-11-28 16:24...|            Braganca|          1|              1444|              1444.0|
|{2025-11-28 16:27...|Regiao Autonoma d...|          1|               580|               580.0|
|{2025-11-28 16:24...|              Leiria|          1|               941|               941.0|
|{2025-11-28 16:24...|              Guarda|          1|               448|               448.0|
|{2025-11-28 16:24...|      Castelo Branco|          1|               428|               428.0|
|{2025-11-28 16:24...|    Viana do Castelo|          1|              1450|              1450.0|
|{2025-11-28 16:27...|             Setubal|          1|              1091|              1091.0|
|{2025-11-28 16:24...|               Evo

#### Advanced Features

Structured Streaming supports ``Joins``. This means that you are able to (I) join a stream with a static DataFrame and (II) join two streams. This can be used to supplement streaming data with another data source.

Here, we will supplement our ``scotsmen`` table with the ``bag_pipes_sales`` table.

In [9]:
# Again, we need to read from Kafka.
# This time, we subscribe to the bagpipes topic
bagpipes_stream = spark.readStream.format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS) \
    .option("subscribe", "bagpipe") \
    .option("startingOffsets", "earliest") \
    .load()    

In [10]:
# Instantiate the schema of the messages received
bagpipes_schema = StructType([
  StructField("district", StringType()),
  StructField("bagpipe_sales", IntegerType()),
  StructField("timestamp", TimestampType())
])

In [11]:
# We select the 'value' column and cast it as a String
bagpipes_json_df = bagpipes_stream.select(
    F.from_json(F.col("value").cast("string"), bagpipes_schema).alias("value"), 
    )

In [12]:
# Convert the JSON to DataFrame and alias columns
bagpipe_df = bagpipes_json_df.select(
    F.col("value.district").alias("sales_district"),
    F.col("value.timestamp").alias("sale_timestamp"),
    F.col("value.bagpipe_sales").alias("bagpipe_sales")
)

In [15]:
# Add watermarking to both streams to handle late data
conversion_stream = new_scotsmen_df.withColumn("conversion_truncated_timestamp", F.date_trunc("minute", new_scotsmen_df["conversion_timestamp"]))
bagpipes_stream = bagpipe_df.withColumn("sales_truncated_timestamp", F.date_trunc("minute", bagpipe_df["sale_timestamp"]))

# Watermark the datasets
conversion_stream = conversion_stream.withWatermark("conversion_truncated_timestamp", "1 minute")
bagpipes_stream = bagpipes_stream.withWatermark("sales_truncated_timestamp", "1 minute")

# Alias the datasets
conversion_stream = conversion_stream.alias("s1")
bagpipes_stream = bagpipes_stream.alias("s2")

# Perform the join between the two windowed streams on 'district' and matching 
# windows by using a functional expression
joined_stream = conversion_stream \
    .join(
        bagpipes_stream,
        F.expr("""
            s1.conversion_district = s2.sales_district AND
            s2.sales_truncated_timestamp >= s1.conversion_truncated_timestamp AND
            s2.sales_truncated_timestamp <= s1.conversion_truncated_timestamp + interval 5 minute
        """)
    ) \
    .select(
        "s2.sales_truncated_timestamp",
        "s1.new_scotsmen",
        "s2.bagpipe_sales",
        "s1.conversion_district"
    )

# Create the window column before aggregation
joined_stream = joined_stream.withColumn("window", F.window("sales_truncated_timestamp", "2 minutes"))

# Aggregate values per district
aggregated_stream = joined_stream \
    .groupBy(
        joined_stream.conversion_district,
        joined_stream.window
    ) \
    .agg(
        F.sum(joined_stream.new_scotsmen).alias("total_new_scotsmen"),
        F.sum(joined_stream.bagpipe_sales).alias("total_bagpipe_sales")
    )

In [17]:
# Output the results to the console for inspection
query = aggregated_stream \
    .writeStream \
    .queryName("stream_stream_join") \
    .outputMode("append") \
    .format("memory") \
    .start()

25/11/28 16:34:47 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-61be1be6-ed85-42c9-91f5-9d044f4bef6c. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
25/11/28 16:34:47 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


25/11/28 16:34:48 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
25/11/28 16:34:48 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


In [19]:
# Showing the results
spark.sql("SELECT * FROM stream_stream_join").show()

+--------------------+--------------------+------------------+-------------------+
| conversion_district|              window|total_new_scotsmen|total_bagpipe_sales|
+--------------------+--------------------+------------------+-------------------+
|              Guarda|{2025-11-28 16:26...|               896|               1058|
|           Vila Real|{2025-11-28 16:26...|              1168|                545|
|             Coimbra|{2025-11-28 16:26...|              1427|                437|
|            Braganca|{2025-11-28 16:26...|              4332|               1245|
|                Beja|{2025-11-28 16:26...|               949|                532|
|    Viana do Castelo|{2025-11-28 16:26...|              1450|                581|
|              Leiria|{2025-11-28 16:26...|              1882|               1025|
|            Santarem|{2025-11-28 16:26...|               479|                 74|
|Regiao Autonoma d...|{2025-11-28 16:26...|               902|                308|
|   

#### Static-Stream Joins

Apart from joining two streams, Spark also supports joining a stream with a static DataFrame. This can be used to supplement streaming data with another data source, such as a lookup table. Here  we will supplement our ``conversation_stream`` with the ``portugal_district_population2022.csv`` table.

In [27]:
population_schema = StructType([
  StructField("district", StringType()),
  StructField("pop", IntegerType())
])

population_df = spark \
    .read \
    .format("csv") \
    .option("header", True) \
    .schema(population_schema) \
    .load("portugal_district_population2022.csv")

In [28]:
# Join the conversion stream with the population data
joined_stream_population = conversion_stream.join(
    population_df,
    conversion_stream.conversion_district == population_df.district,
    "inner" 
) \
    .withColumn(
        "conversions_per_pop", F.col("new_scotsmen") / F.col("pop")
    )

# Select the columns you need
result_stream = joined_stream_population.select(
    "conversion_truncated_timestamp",
    "new_scotsmen",
    "conversion_district",
    "pop",
    "conversions_per_pop"
)

In [30]:
# Output the results to the console for testing
query = result_stream \
    .writeStream \
    .queryName("prop_pop_converted") \
    .outputMode("append") \
    .format("memory") \
    .start()

25/11/28 16:28:29 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-c916494a-2c59-4080-9505-18bf417ab150. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
25/11/28 16:28:29 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


IllegalArgumentException: Cannot start query with name prop_pop_converted as a query with that name is already active in this SparkSession

In [31]:
# Define the window duration and slide duration
window_duration = "1 hour"
slide_duration = "10 minutes"

# SQL query to compute the cumulative sum of the ratio
sql_query = f"""
SELECT
    window.start AS window_start,
    window.end AS window_end,
    SUM(conversions_per_pop) AS cumulative_ratio,
    conversion_district
FROM (
    SELECT
        conversions_per_pop,
        window(current_timestamp(), '{window_duration}', '{slide_duration}') AS window,
        conversion_district
    FROM prop_pop_converted
)
GROUP BY conversion_district, window
ORDER BY window_start
"""

# Execute the SQL query
result_df = spark.sql(sql_query)

# Show the result
result_df.show()

+-------------------+-------------------+--------------------+--------------------+
|       window_start|         window_end|    cumulative_ratio| conversion_district|
+-------------------+-------------------+--------------------+--------------------+
|2025-11-28 15:30:00|2025-11-28 16:30:00|0.006176492652390994|Regiao Autonoma d...|
|2025-11-28 15:30:00|2025-11-28 16:30:00|0.002014389721260...|              Leiria|
|2025-11-28 15:30:00|2025-11-28 16:30:00| 0.01077543762906611|      Castelo Branco|
|2025-11-28 15:30:00|2025-11-28 16:30:00|0.006244078890707088|    Viana do Castelo|
|2025-11-28 15:30:00|2025-11-28 16:30:00|0.008524530104086932|               Evora|
|2025-11-28 15:30:00|2025-11-28 16:30:00|0.003170302096566...|             Setubal|
|2025-11-28 15:30:00|2025-11-28 16:30:00|0.015580166979042426|                Beja|
|2025-11-28 15:30:00|2025-11-28 16:30:00|0.003171434436964201|              Guarda|
|2025-11-28 15:30:00|2025-11-28 16:30:00|0.005887253759984...|Regiao Autonom

#### Simulating Streaming Datasets

It is also possible to "simulate" a streaming dataset by reading from a directory of CSV files. This can be useful for testing and debugging purposes. The dataset contains individual files that can be read as a batch of data.

In [20]:
# Define the weather schema
schema = StructType([
    StructField("timestamp", TimestampType(), True),
    StructField("min_temperature", DoubleType(), True),
    StructField("max_temperature", DoubleType(), True),
    StructField("precipitation", DoubleType(), True)
])

# Path to the directory containing the CSV files
# Note that this must be a directory and not an individual file
input_path = "weather"

NUM_FILES_PER_TRIGGER = 3

# Read the streaming DataFrame from the directory
streaming_df = spark.readStream \
    .option("maxFilesPerTrigger", NUM_FILES_PER_TRIGGER) \
    .option("header", "true") \
    .format("csv") \
    .schema(schema) \
    .load(input_path)

# Define the query to process the streaming data
# It is possible to set the batch size 
# to control how frequently the streaming query processes new data.
# This is done using the trigger option in the writeStream method. 
# The trigger option allows you to specify the processing time interval, 
# which determines the batch size.
# maxFilesPerTrigger is the maximum number of files that will be
# processed in a single trigger.
query = streaming_df.writeStream \
    .trigger(processingTime='10 seconds') \
    .option("maxFilesPerTrigger", 5) \
    .outputMode("append") \
    .queryName("weather") \
    .format("memory") \
    .start()

25/11/28 16:48:09 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-445c57d4-c3a5-4b16-b6c4-cf2cd4b7bf95. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
25/11/28 16:48:09 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [24]:
# We can now query the static dataset just like we did before
spark.sql(
    """SELECT max_temperature, min_temperature, timestamp 
    FROM weather ORDER BY timestamp 
    DESC
    """
).show()

+------------------+------------------+-------------------+
|   max_temperature|   min_temperature|          timestamp|
+------------------+------------------+-------------------+
|26.048084828624965|  12.9269061041685|2024-01-03 03:51:00|
| 36.74800636236662|22.509850132528094|2024-01-03 03:50:00|
|30.798064582878645|17.997721383715202|2024-01-03 03:45:00|
|25.303320710991315|13.487460281537103|2024-01-03 03:44:00|
|24.215109775889218|10.948218440431674|2024-01-03 03:10:00|
|37.520348875559264|27.078055554874556|2024-01-03 02:33:00|
|23.547397695431535|17.112853708317733|2024-01-03 02:27:00|
|32.403072554793674|22.027655550970376|2024-01-03 01:32:00|
| 35.25158051468364|22.934208698361616|2024-01-03 00:45:00|
+------------------+------------------+-------------------+

